In [31]:
import pandas as pd
from pathlib import Path
import os


DATA_DIR = Path('../data/raw')
print(list(DATA_DIR.glob('*.XPT')))


[PosixPath('../data/raw/DEMO_L.XPT'), PosixPath('../data/raw/BMX_L.XPT'), PosixPath('../data/raw/DIQ_L.XPT'), PosixPath('../data/raw/GHB_L.XPT')]


In [32]:
# load files
demo_l = pd.read_sas(DATA_DIR / 'DEMO_L.xpt', format='xport', encoding='utf-8')
ghb_l  = pd.read_sas(DATA_DIR / 'GHB_L.xpt',  format='xport', encoding='utf-8')
diq_l  = pd.read_sas(DATA_DIR / 'DIQ_L.xpt',  format='xport', encoding='utf-8')
bmx_l  = pd.read_sas(DATA_DIR / 'BMX_L.xpt',  format='xport', encoding='utf-8')

In [33]:
# merge data
df = demo_l.merge(ghb_l, on='SEQN', how='left')
df = df.merge(diq_l,  on='SEQN', how='left')
df = df.merge(bmx_l,  on='SEQN', how='left')
df['cycle'] = '2021-2023'

print(f"Merged shape: {df.shape}")

Merged shape: (11933, 59)


In [34]:
key_cols = ['SEQN', 'cycle', 'DIQ010', 'RIDRETH3', 'LBXGH', 'BMXBMI', 'RIDAGEYR', 'RIAGENDR']

print(df.shape)

print("\nKey column names present:")
print([c for c in key_cols if c in df.columns])

print("\nDiabetes diagnosis (DIQ010) value counts:")
print(df['DIQ010'].value_counts(dropna=False))

print("\nRace/ethnicity (RIDRETH3) value counts:")
print(df['RIDRETH3'].value_counts(dropna=False))

print("\nHbA1c (LBXGH) summary:")
print(df['LBXGH'].describe())

print("\n BMI (BMXBMI) summary:")
print(df['BMXBMI'].describe())

(11933, 59)

Key column names present:
['SEQN', 'cycle', 'DIQ010', 'RIDRETH3', 'LBXGH', 'BMXBMI', 'RIDAGEYR', 'RIAGENDR']

Diabetes diagnosis (DIQ010) value counts:
DIQ010
2.0    10371
1.0     1081
3.0      284
NaN      193
9.0        4
Name: count, dtype: int64

Race/ethnicity (RIDRETH3) value counts:
RIDRETH3
3.0    6217
4.0    1597
2.0    1373
1.0    1117
7.0     948
6.0     681
Name: count, dtype: int64

HbA1c (LBXGH) summary:
count    6715.000000
mean        5.709471
std         1.054899
min         3.200000
25%         5.200000
50%         5.500000
75%         5.800000
max        17.100000
Name: LBXGH, dtype: float64

 BMI (BMXBMI) summary:
count    8471.000000
mean       27.246665
std         8.137781
min        11.100000
25%        21.600000
50%        26.400000
75%        31.700000
max        74.800000
Name: BMXBMI, dtype: float64


In [35]:
import numpy as np

# Recode: 1 = diabetes, 0 = no diabetes, drop borderline/refused for now
df['diabetes'] = np.where(df['DIQ010'] == 1, 1,
                 np.where(df['DIQ010'] == 2, 0, np.nan))

print("Target variable distribution")
print(df['diabetes'].value_counts(dropna=False))
print(f"\nClass balance: {df['diabetes'].mean():.1%} diabetic")

# Check across race/ethnicity subgroups
# RIDRETH3: 1=Mexican American, 2=Other Hispanic, 3=Non-Hispanic White,
#            4=Non-Hispanic Black, 6=Non-Hispanic Asian, 7=Other/Multi
race_labels = {1:'Mexican American', 2:'Other Hispanic', 3:'NH White',
               4:'NH Black', 6:'NH Asian', 7:'Other/Multi'}
df['race_label'] = df['RIDRETH3'].map(race_labels)

print("\n Diabetes rate by race/ethnicity:")
print(df.groupby('race_label')['diabetes'].agg(['mean','sum','count']).round(3))

Target variable distribution
diabetes
0.0    10371
1.0     1081
NaN      481
Name: count, dtype: int64

Class balance: 9.4% diabetic

 Diabetes rate by race/ethnicity:
                   mean    sum  count
race_label                           
Mexican American  0.084   90.0   1070
NH Asian          0.077   49.0    633
NH Black          0.124  191.0   1536
NH White          0.091  548.0   5997
Other Hispanic    0.088  115.0   1309
Other/Multi       0.097   88.0    907


In [ ]:
print("Missing value counts for key columns:")
print(df[key_cols + ['diabetes']].isnull().sum())

os.makedirs('../data/processed', exist_ok=True)

# Save as CSV instead of parquet
df.to_csv('../data/processed/merged_nhanes.csv', index=False)
print("Saved to data/processed/merged_nhanes.csv")


=== Missing value counts for key columns ===
SEQN           0
cycle          0
DIQ010       193
RIDRETH3       0
LBXGH       5218
BMXBMI      3462
RIDAGEYR       0
RIAGENDR       0
diabetes     481
dtype: int64
Saved to data/processed/merged_nhanes.csv
